## 0) DOE 데이터 재생성 (선택)

새 DOE 데이터를 만들 때만 실행합니다.

- 실행 환경: **호스트 Motor-CAD** (Docker 아님)
- 생성 결과:
  - `case_NNNN/*.mot` : 케이스별 기준 모델 복사본
  - `case_NNNN/FEResultsData/` : Motor-CAD 원본 결과 루트
  - `case_NNNN/doe_condition.json` : DOE 변수/적용값
  - `case_NNNN/postproc/*.txt` : 메인 DOE 배치 산출물
  - `case_NNNN/postproc/*.h5` : 필요 시 0-D 후속 셀에서 생성

권장 순서:
1. 0-B 설정 셀에서 경로/샘플 수/단계를 지정
2. `RUN_DOE=True`로 변경
3. 0-C 실행 셀 실행 (`solve + txt`)
4. 학습까지 이어갈 때만 `RUN_H5_EXPORT=True`로 바꿔 0-D 실행

**중간 중단 후 복구:**
- 0-E-1 셀로 상태 점검 (`.mes` 있음 / `.txt` 없음 케이스 식별)
- 0-E-2 셀에서 `RUN_REPAIR=True`로 바꿔 `.txt` 재내보내기

In [ ]:

# 0-B) DOE 설정 셀 (Host Motor-CAD)
import json
import os
import sys
import importlib
from pathlib import Path



import ansys.motorcad.core as pymotorcad

ROOT = Path.cwd()
EMACH_ROOT = ROOT / "eMach"
if str(EMACH_ROOT) not in sys.path:
    sys.path.insert(0, str(EMACH_ROOT))

from tools.pyutils.sweep import DOEAxis, DOEPoint, build_doe_lhs
import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
importlib.reload(_doe_batch)
from tools.motorCAD.pyMCAD.doe_batch import doe_batch_run, doe_h5_batch_from_txt

# ----- 사용자 설정 -----
BASE_MOT = r"D:\KDH\Sim_4SolverX\TestCAD1.mot"
DOE_OUT = Path(r"D:\KDH\Sim_4SolverX\DOE4TrainingData")
DOE_OUT.mkdir(parents=True, exist_ok=True)

N_SAMPLES = 40
LHS_SEED = 42
LHS_CRITERION = "maximin"

PHASES = ["solve", "export_txt"]
# backward compatibility가 필요하면 아래 legacy 경로 사용:
# PHASES = ["solve", "export"]

FIRST_STEP = 1
FINAL_STEP = 45
MAG_H5_MESH_COORDS = "by_step_moving_nodes"
MAG_COLUMNS = "RegCode,Bx,By,A,J,Je"
PLOT_MODE = "none"

# 병렬 실행 워커 수 (1=직렬, 2 이상=병렬)
# Windows + Motor-CAD COM 환경에서는 2부터 시작하는 것을 권장
PARALLEL_WORKERS = 8

# 안전 스위치
RUN_DOE = False
RUN_H5_EXPORT = True

print(f"ROOT={ROOT}")
print(f"EMACH_ROOT={EMACH_ROOT}")
print(f"BASE_MOT={BASE_MOT}")
print(f"DOE_OUT={DOE_OUT}")
print(f"PHASES={PHASES}, N_SAMPLES={N_SAMPLES}, RUN_DOE={RUN_DOE}")
print(f"RUN_H5_EXPORT={RUN_H5_EXPORT}")
print(f"PARALLEL_WORKERS={PARALLEL_WORKERS}")


In [ ]:

# 0-C) 실행 셀: DOE 데이터 생성 (solve + txt-only)
#
# [첫 실행]
#   → Motor-CAD 접속 → build_doe_lhs → doe_grid.json 저장 → 전체 케이스 실행
#
# [재실행 — 0-E-1 통합]
#   → doe_grid.json 로드 (build_doe_lhs 생략)
#   → doe_scan_status 로 상태 점검
#   → needs_solve / not-started 케이스만 doe_batch_run
#   → needs_txt 케이스는 doe_repair_missing_txt 로 txt 재내보내기 (병렬 지원)
#   → 모두 complete 이면 신규 실행 없이 요약만 출력

# ── helpers ──────────────────────────────────────────────────────────────────
def _get_var(mc, name):
    """Motor-CAD get_variable 반환값 언래핑.
    enable_success_variable=True 환경에서는 (returncode, value) 튜플을 반환하므로
    value만 추출합니다."""
    v = mc.get_variable(name)
    return v[1] if isinstance(v, tuple) and len(v) == 2 and isinstance(v[0], int) else v


def _save_doe_grid(grid, path, meta=None):
    """DOEPoint 리스트를 JSON으로 직렬화해 저장합니다."""
    data = {
        "meta": meta or {},
        "points": [
            {"geometry": pt.geometry, "electrical": pt.electrical, "index": pt.index}
            for pt in grid
        ],
    }
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def _load_doe_grid(path):
    """JSON에서 DOEPoint 리스트를 복원합니다."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    return [
        DOEPoint(geometry=p["geometry"], electrical=p["electrical"], index=p["index"])
        for p in data["points"]
    ]

# ── 실행 본체 ────────────────────────────────────────────────────────────────
DOE_GRID_JSON = DOE_OUT / "doe_grid.json"

if not RUN_DOE:
    print("RUN_DOE=False 입니다. 설정 확인 후 True로 바꿔 실행하세요.")
else:
    importlib.reload(_doe_batch)
    from tools.motorCAD.pyMCAD.doe_batch import (
        doe_batch_run, doe_scan_status, doe_repair_missing_txt,
    )

    # ── Step 1: DOE 그리드 로드 / 신규 생성 ─────────────────────────────
    if DOE_GRID_JSON.exists():
        doe_grid = _load_doe_grid(DOE_GRID_JSON)
        print(f"[0-C] 기존 DOE 그리드 로드: {len(doe_grid)}개 포인트 ({DOE_GRID_JSON.name})")
        print("      → build_doe_lhs 생략 (Motor-CAD 접속 지연)")
    else:
        # 기준값 읽기용 Motor-CAD 접속
        try:
            mc = pymotorcad.MotorCAD(open_new_instance=False)
            print("Motor-CAD: existing instance connected")
        except Exception as _e0:
            print(f"[warn] existing instance connect failed: {_e0}")
            print("Motor-CAD: launching new instance...")
            mc = pymotorcad.MotorCAD(open_new_instance=True)

        mc.load_from_file(BASE_MOT)
        rb_base = float(_get_var(mc, "Ratio_Bore"))
        rsd_base = float(_get_var(mc, "Ratio_SlotDepth_ParallelSlot"))

        all_axes = [
            DOEAxis("Ratio_Bore", round(rb_base * 0.90, 4), round(rb_base * 1.10, 4), steps=0),
            DOEAxis("Ratio_SlotDepth_ParallelSlot", round(rsd_base * 0.85, 4), round(rsd_base * 1.15, 4), steps=0),
            DOEAxis("PeakCurrent", 10.0, 650.53, steps=0),
            DOEAxis("PhaseAdvance", 0.0, 90.0, steps=0),
        ]
        doe_grid = build_doe_lhs(
            axes=all_axes, n_samples=N_SAMPLES, seed=LHS_SEED, criterion=LHS_CRITERION,
        )
        _save_doe_grid(doe_grid, DOE_GRID_JSON, meta={
            "n_samples": N_SAMPLES, "seed": LHS_SEED, "criterion": LHS_CRITERION,
            "axes": [{"name": ax.name, "min": ax.min_val, "max": ax.max_val} for ax in all_axes],
        })
        print(f"[0-C] 신규 DOE 그리드 생성: {len(doe_grid)}개 포인트 → {DOE_GRID_JSON.name}")

    # ── Step 2: 상태 점검 (0-E-1 통합) ──────────────────────────────────
    scan = doe_scan_status(DOE_OUT, verbose=True)
    _scanned_idx = {c["index"] for c in scan["cases"]}
    _not_started = [pt.index for pt in doe_grid if pt.index not in _scanned_idx]
    _solve_idx = _not_started + scan["solve_needed"]  # 미시작 + 해석 미완료
    _repair_idx = scan["repair_needed"]  # mes 있음, txt 없음

    print(json.dumps({
        "summary": scan["summary"],
        "not_started": len(_not_started),
        "solve_needed": len(_solve_idx),
        "repair_needed": len(_repair_idx),
    }, indent=2, ensure_ascii=False))

    if not _solve_idx and not _repair_idx:
        # 모든 케이스 완료 — 신규 실행 없음
        print("✓ 모든 케이스가 완료 상태입니다. 신규 실행 없음.")
        _txt_ready = scan["summary"].get("complete", 0)
        manifest = {}
    else:
        # Motor-CAD 인스턴스 확보 (아직 없으면 접속)
        if "mc" not in globals():
            try:
                mc = pymotorcad.MotorCAD(open_new_instance=False)
                print("Motor-CAD: existing instance connected")
            except Exception as _e1:
                print(f"[warn] {_e1}")
                mc = pymotorcad.MotorCAD(open_new_instance=True)
                print("Motor-CAD: launching new instance...")
            mc.load_from_file(BASE_MOT)

        # ── Step 3-a: 미시작 / needs_solve 케이스 해석 ───────────────
        if _solve_idx:
            _solve_set = set(_solve_idx)
            _solve_grid = [pt for pt in doe_grid if pt.index in _solve_set]
            print(f"[0-C] solve → {len(_solve_grid)}개 케이스 실행...")
            manifest = doe_batch_run(
                mc, _solve_grid,
                base_mot=BASE_MOT, doe_out_root=DOE_OUT, phases=PHASES,
                first_step=FIRST_STEP, final_step=FINAL_STEP,
                mag_columns=MAG_COLUMNS, mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
                plot_mode=PLOT_MODE, parallel_workers=PARALLEL_WORKERS,
            )
        else:
            manifest = {}

        # ── Step 3-b: needs_txt 케이스 txt 재내보내기 ────────────────
        if _repair_idx:
            print(
                f"[0-C] needs_txt 복구 → {len(_repair_idx)}개 케이스 "
                f"(parallel_workers={PARALLEL_WORKERS})..."
            )
            _repair = doe_repair_missing_txt(
                mc, DOE_OUT, case_indices=_repair_idx,
                first_step=FIRST_STEP, final_step=FINAL_STEP,
                mag_columns=MAG_COLUMNS, mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
                plot_mode=PLOT_MODE, parallel_workers=PARALLEL_WORKERS,
                verbose=True,
            )
            print(json.dumps({
                "repair_repaired": _repair["repaired"],
                "repair_failed": _repair["failed"],
            }, indent=2, ensure_ascii=False))

        # 최종 완료 수 재집계
        _final_scan = doe_scan_status(DOE_OUT, verbose=False)
        _txt_ready = _final_scan["summary"].get("complete", 0)

    print(json.dumps({
        "base_mot": BASE_MOT,
        "doe_out": str(DOE_OUT),
        "doe_grid_json": str(DOE_GRID_JSON),
        "n_cases": len(doe_grid),
        "txt_ready_cases": _txt_ready,
        "failed": len(manifest.get("failed", [])),
    }, indent=2, ensure_ascii=False))


### 0-D) 선택 셀: TXT -> H5 변환

학습/추론 섹션으로 바로 이어갈 때만 실행합니다.

- 실행 환경: 호스트 Python만 필요
- 입력: 0-C에서 생성된 `case_NNNN/postproc/Mag_*.txt`
- 출력: `case_NNNN/postproc/Mag_*.h5` + `doe_manifest.json`의 `h5_paths` 갱신

In [ ]:
if not RUN_H5_EXPORT:
    print("RUN_H5_EXPORT=False 입니다. H5가 필요할 때만 True로 바꿔 실행하세요.")
else:
    h5_summary = doe_h5_batch_from_txt(
        DOE_OUT,
        mag_h5_mesh_coords=MAG_H5_MESH_COORDS,
        verbose=True,
    )

    h5_ready = sum(
        1 for case in h5_summary.get("cases", [])
        if not case.get("error")
    )

    print(json.dumps({
        "doe_out": str(DOE_OUT),
        "requested_cases": len(h5_summary.get("cases", [])),
        "h5_ready_cases": h5_ready,
    }, indent=2, ensure_ascii=False))


### 0-E) 수동 복구: DOE 상태 재점검 + TXT 재내보내기

0-C 재실행만으로 복구되지 않는 경우(예: `mc` 접속 없이 상태만 확인하고 싶을 때)에 사용합니다.

> **일반 복구 흐름:** 0-C 셀 재실행만으로 충분합니다 (`doe_grid.json` 이 있으면 자동 상태 점검 → 미완성 케이스만 실행).

**0-E 점검 항목:**
| 상태 | 의미 | 자동 조치 (0-C 재실행 시) |
|------|------|--------------------------|
| `complete` | `.mes` + `.txt` 모두 있음 | 없음 |
| `needs_txt` | `.mes` 있음, `.txt` 없음 | `doe_repair_missing_txt` 자동 호출 |
| `needs_solve` | `.mes` 없음 (해석 미완료) | `doe_batch_run` 자동 호출 |
| `empty` | 초기화 흔적 없음 | 무시 |

**0-E 수동 실행 순서 (0-C 재실행이 어려울 때):**
1. `0-E-1` 상태 점검 셀 실행 → 복구 대상 인덱스 확인
2. `RUN_REPAIR=True`로 변경 후 `0-E-2` 복구 셀 실행


In [ ]:
# 0-E-1) DOE 상태 점검
# 이 셀은 Motor-CAD 없이 실행 가능합니다.
import json
import importlib
from pathlib import Path

# 0-B 셀을 먼저 실행하지 않은 경우를 위한 독립 경로 설정
_doe_out_check = globals().get("DOE_OUT", Path(r"D:\KDH\Sim_4SolverX\DOE4TrainingData"))

import tools.motorCAD.pyMCAD.doe_batch as _doe_batch
importlib.reload(_doe_batch)
from tools.motorCAD.pyMCAD.doe_batch import doe_scan_status

scan_result = doe_scan_status(_doe_out_check, verbose=True)

# 결과 요약 출력
print(json.dumps({
    "doe_out": str(_doe_out_check),
    "summary": scan_result["summary"],
    "repair_needed_count": len(scan_result["repair_needed"]),
    "solve_needed_count": len(scan_result["solve_needed"]),
    "repair_needed_indices": scan_result["repair_needed"][:20],   # 최대 20개만 표시
    "solve_needed_indices": scan_result["solve_needed"][:20],
}, indent=2, ensure_ascii=False))


In [ ]:

# 0-E-2) TXT 재내보내기 복구 셀 (mes 있음 + txt 없는 케이스)
# 안전 스위치 — True로 변경해야 실행됩니다.
RUN_REPAIR = True

if not RUN_REPAIR:
    print("RUN_REPAIR=False 입니다. 복구할 케이스가 있으면 True로 바꿔 실행하세요.")
    print(f"  복구 대상 케이스: {scan_result.get('repair_needed', [])}")
else:
    import importlib
    import tools.motorCAD.pyMCAD.doe_batch as _doe_batch_repair
    importlib.reload(_doe_batch_repair)
    from tools.motorCAD.pyMCAD.doe_batch import doe_repair_missing_txt

    # Motor-CAD 인스턴스 준비 (0-C에서 mc가 이미 있으면 재사용)
    if "mc" not in globals():
        import ansys.motorcad.core as pymotorcad
        try:
            mc = pymotorcad.MotorCAD(open_new_instance=False)
            print("Motor-CAD: existing instance connected")
        except Exception as _e:
            print(f"[warn] {_e}")
            mc = pymotorcad.MotorCAD(open_new_instance=True)
            print("Motor-CAD: new instance launched")

    # 복구 대상 인덱스 (None = 점검에서 발견된 needs_txt 전체)
    REPAIR_CASE_INDICES = scan_result.get("repair_needed") or None
    REPAIR_PARALLEL_WORKERS = int(globals().get("PARALLEL_WORKERS", 1))

    repair_summary = doe_repair_missing_txt(
        mc,
        _doe_out_check,
        case_indices=REPAIR_CASE_INDICES,
        first_step=globals().get("FIRST_STEP", 1),
        final_step=globals().get("FINAL_STEP", 45),
        mag_columns=globals().get("MAG_COLUMNS", "RegCode,Bx,By,A,J,Je"),
        mag_h5_mesh_coords=globals().get("MAG_H5_MESH_COORDS", "by_step_moving_nodes"),
        plot_mode="none",
        parallel_workers=REPAIR_PARALLEL_WORKERS,
        dry_run=False,
        verbose=True,
    )

    print(json.dumps({
        "doe_out": str(_doe_out_check),
        "parallel_workers": REPAIR_PARALLEL_WORKERS,
        "repaired": repair_summary["repaired"],
        "failed": repair_summary["failed"],
        "skipped": repair_summary["skipped"],
    }, indent=2, ensure_ascii=False))


# Phase 1 Tutorial — SymMGN PBC 전체 검증

이 노트북은 Phase 1 (Static SymMGN with PBC) 완료 증거를 생성합니다.

**핵심 원칙:**
- 로컬 커널은 오케스트레이션/시각화만 수행
- 모델 학습/추론은 `motor_compare` 컨테이너 내부에서 실행

**실행 순서:**
| Part | 내용 | 셀 |
|------|------|----|
| 0 | (선택) DOE 데이터 재생성 + H5 후속 변환 | 0-B ~ 0-D |
| A | 환경 + Docker + GPU | 1-4 |
| B | Contract / PBC / Overfit 검증 | 5-7 |
| C | **PBC 경계 가시화** (학습 전 확인) | 8 |
| D | 3-case DOE smoke test | 9-10 |
| E | **Full 40-case 학습** | 11 |
| F | **Full 40-case 추론 + 시각화** | 12-13 |
| G | Phase 1 완료 Evidence | 14

## 1) 환경 설정 및 라이브러리 임포트

로컬에서는 실행 제어/결과 파싱만 수행합니다.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd()
CONTAINER_NAME = os.environ.get("PHYSICSNEMO_CONTAINER", "motor_compare")
LOG_DIR = ROOT / "logs" / "tutorial"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"CONTAINER_NAME={CONTAINER_NAME}")
print(f"LOG_DIR={LOG_DIR}")

## 2) Docker 실행 헬퍼

학습/추론/테스트는 모두 컨테이너 내부에서 실행합니다.

In [ ]:
import shutil
from datetime import datetime


def run_local(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    print(f"[local] {cmd}")
    return subprocess.run(cmd, shell=True, text=True, capture_output=True, check=check)


def run_docker(cmd: str, check: bool = True) -> subprocess.CompletedProcess:
    # Use double-quoted bash -lc payload to avoid breaking on single quotes in Python code.
    payload = f"cd /workspace/app && {cmd}"
    payload = payload.replace("\\", "\\\\").replace('"', '\\"')
    full_cmd = f'docker exec {CONTAINER_NAME} bash -lc "{payload}"'
    print(f"[docker] {cmd}")
    return run_local(full_cmd, check=check)


def save_log(name: str, cp: subprocess.CompletedProcess) -> Path:
    path = LOG_DIR / name
    text = []
    text.append(f"$ returncode={cp.returncode}\n")
    if cp.stdout:
        text.append("\n[stdout]\n")
        text.append(cp.stdout)
    if cp.stderr:
        text.append("\n[stderr]\n")
        text.append(cp.stderr)
    path.write_text("".join(text), encoding="utf-8")
    print(f"saved: {path}")
    return path


def backup_npz_dir(npz_dir: Path, label: str = "") -> "Path | None":
    """NPZ 결과 디렉토리를 타임스탬프 백업으로 이동한다.

    코드(모델·손실·채널·정규화 등)를 수정한 뒤 추론을 재실행하기 전에
    반드시 호출해 기존 결과를 보존한다.

    백업 경로:
        <npz_dir>/../backups/<dirname>_YYYYMMDD_HHMMSS[_label]/

    Args:
        npz_dir: NPZ 파일이 있는 디렉토리 (Path 또는 str).
        label:   백업 디렉토리 이름에 붙일 짧은 설명 (예: "before_4ch").

    Returns:
        백업 디렉토리 Path (NPZ가 없었으면 None).

    Policy:
        - 학습/추론 코드(contracts.py, motor_dataset.py, infer_*.py 등)를
          수정했다면 재실행 전 이 함수를 먼저 호출한다.
        - checkpoint 파일(.pt)도 교체될 경우 수동으로 같은 backups/ 폴더에
          복사해 둔다.
        - 백업은 덮어쓰지 않는다 (타임스탬프로 고유 디렉토리 생성).
    """
    npz_dir = Path(npz_dir)
    if not npz_dir.exists():
        print(f"[backup] {npz_dir} 없음 — 건너뜀")
        return None

    existing_npz = sorted(npz_dir.glob("*.npz"))
    if not existing_npz:
        print(f"[backup] {npz_dir} 에 NPZ 없음 — 건너뜀")
        return None

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    suffix = f"_{label}" if label else ""
    backup_dir = npz_dir.parent / "backups" / f"{npz_dir.name}_{ts}{suffix}"
    backup_dir.mkdir(parents=True, exist_ok=True)

    for f in existing_npz:
        shutil.move(str(f), str(backup_dir / f.name))

    print(f"[backup] {len(existing_npz)}개 NPZ → {backup_dir}")
    return backup_dir


print("helpers ready: run_local / run_docker / save_log / backup_npz_dir")


## 3) GPU-enabled Docker Compose 기동

GPU, IPC, ulimit 설정이 적용되도록 컨테이너를 재생성합니다.

In [ ]:
cp_compose_up = run_local("docker compose up -d --force-recreate", check=False)
save_log("00_compose_up.log", cp_compose_up)
print(cp_compose_up.stdout)
if cp_compose_up.stderr.strip():
    print(cp_compose_up.stderr)
if cp_compose_up.returncode != 0:
    raise RuntimeError("GPU-enabled docker compose up 에 실패했습니다. compose 설정과 Docker GPU 런타임을 확인하세요.")

## 4) GPU / PyTorch 확인 (컨테이너 내부)

In [ ]:
cp_nvidia_smi = run_docker("nvidia-smi", check=False)
save_log("01_nvidia_smi.log", cp_nvidia_smi)
print(cp_nvidia_smi.stdout or cp_nvidia_smi.stderr)

cp_torch = run_docker(
    "python -c \"import torch; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('device_count', torch.cuda.device_count()); print('cuda_version', torch.version.cuda)\"",
    check=False,
 )
save_log("01_torch_env.log", cp_torch)
print(cp_torch.stdout)
if cp_torch.stderr.strip():
    print(cp_torch.stderr)
if cp_nvidia_smi.returncode != 0 or cp_torch.returncode != 0 or "cuda False" in cp_torch.stdout:
    raise RuntimeError(
        "GPU가 컨테이너에 노출되지 않았습니다. 01_nvidia_smi.log 와 01_torch_env.log를 확인하세요."
    )

## 5) Contract / PBC / Overfit 검증 게이트

Phase 1 계약 경계 테스트 → PBC 경계/계약 호스트 테스트 → Overfit-Single 게이트 → PBC bundle 테스트

In [ ]:
cp_contract = run_docker(
    "PYTHONPATH=/workspace/app pytest -q tests/test_phase1_contract_boundaries.py",
    check=False,
 )
save_log("02_contract_tests.log", cp_contract)
print(cp_contract.stdout)
if cp_contract.stderr.strip():
    print(cp_contract.stderr)
if cp_contract.returncode != 0:
    raise RuntimeError("Contract 경계 테스트 실패. 로그(02_contract_tests.log)를 확인하세요.")

In [ ]:
import subprocess

# torch 없이 돌아가는 경계/계약 테스트는 호스트 venv에서 허용
# (phase1_static.pbc_boundary, pbc_contracts 는 scipy만 필요)
py_exec = r"c:/Users/moa/.ansys_python_venvs/PyMotorEnv_310/Scripts/python.exe"
cp_pbc_host_tests = subprocess.run(
    [
        py_exec, "-m", "pytest", "-q",
        "tests/test_phase1_pbc_boundary.py",
        "tests/test_phase1_pbc_contracts.py",
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(cp_pbc_host_tests.stdout)
if cp_pbc_host_tests.returncode != 0:
    print("[stderr]", cp_pbc_host_tests.stderr[:600])
print("exit_code:", cp_pbc_host_tests.returncode)

In [ ]:
import subprocess, json, textwrap

OVERFIT_SCRIPT = textwrap.dedent("""
import sys, json, tempfile, subprocess
import numpy as np
from pathlib import Path

try:
    angles = np.deg2rad(np.array([0.0, 22.5, 45.0], dtype=np.float64))
    inner = np.stack([np.cos(angles), np.sin(angles)], axis=1).astype(np.float32)
    pos = inner[None]
    node_type = np.ones((1, 3, 1), dtype=np.float32)
    edges = np.array([[0, 1], [1, 2], [2, 0], [1, 0], [2, 1], [0, 2]], dtype=np.int64)
    ie = edges.T[None]
    pe = np.zeros((1, 2, 0), dtype=np.int64)
    pa = np.zeros((1, 0, 1), dtype=np.float32)
    y = np.random.RandomState(42).randn(1, 3, 5).astype(np.float32)

    with tempfile.TemporaryDirectory() as td:
        npz_path = Path(td) / "overfit.npz"
        np.savez(
            npz_path,
            pos=pos,
            node_type_onehot=node_type,
            interior_edge_index=ie,
            pbc_edge_index=pe,
            pbc_edge_attr=pa,
            y=y,
        )

        r = subprocess.run(
            [sys.executable, "-m", "phase1_static.train",
             "--input-format", "npz", "--data", str(npz_path),
             "--overfit-single", "--epochs", "150",
             "--lr", "1e-2", "--hidden-dim", "32",
             "--seed", "42", "--overfit-target", "1e-2"],
            capture_output=True, text=True, cwd="/workspace/app",
        )
        combined = r.stdout + r.stderr
        epoch_lines = [l for l in combined.splitlines() if "epoch=" in l]
        overfit_lines = [l for l in combined.splitlines() if "Overfit" in l or "overfit" in l]
        print(json.dumps({
            "returncode": r.returncode,
            "last_epoch": epoch_lines[-1] if epoch_lines else "",
            "overfit_gate": overfit_lines[-1] if overfit_lines else "",
        }))
except Exception as exc:
    import traceback
    print(json.dumps({"error": str(exc), "tb": traceback.format_exc()[-400:]}))
""")

cp_overfit = subprocess.run(
    ["docker", "exec", CONTAINER_NAME, "python", "-c", OVERFIT_SCRIPT],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)

print("=== Overfit-Single (Docker) ===")
for line in cp_overfit.stdout.splitlines():
    try:
        d = json.loads(line)
        print(json.dumps(d, indent=2, ensure_ascii=False))
    except Exception:
        print(line)
print("exit_code:", cp_overfit.returncode)

In [ ]:
import subprocess

# pbc_bundle 보강 테스트 + 기존 PBC 테스트 전체 확인
py_exec = r"c:/Users/moa/.ansys_python_venvs/PyMotorEnv_310/Scripts/python.exe"
cp_all_pbc = subprocess.run(
    [
        py_exec, "-m", "pytest", "-q",
        "tests/test_phase1_pbc_bundle.py",
        "tests/test_phase1_pbc_boundary.py",
        "tests/test_phase1_pbc_contracts.py",
        "tests/test_phase1_pbc_pairing.py",
        "tests/test_phase1_pbc_prior.py",
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(cp_all_pbc.stdout)
if cp_all_pbc.returncode != 0:
    print("[stderr]", cp_all_pbc.stderr[:600])
print("exit_code:", cp_all_pbc.returncode)

## 8) PBC 경계 가시화 — 학습 전 확인

실제 DOE 메쉬에서 추출한 master/slave 경계선과 PBC 엣지를 시각화합니다.
학습 전에 PBC topology가 올바르게 구성되었는지 **반드시** 눈으로 확인합니다.

표시 내용:
- 전체 노드 분포 (1/8 섹터)
- master / slave boundary line
- 실제 pbc_edge overlay (anti-periodic, edge_attr=-1.0)
- case별 match ratio, pair 수, group labels

In [ ]:
import importlib
import json

from IPython.display import display

pbc_boundary_module = importlib.import_module(
    "phase1_static.pbc_boundary"
 )
pbc_candidate_module = importlib.import_module(
    "postproc_interop.model.PBCBoundaryCandidate"
 )
pbc_case_module = importlib.import_module(
    "postproc_interop.model.PBCVisualizationCase"
 )
pbc_module = importlib.import_module("postproc_interop.pbc")
pbc_bridge_module = importlib.import_module(
    "postproc_interop.bridges.MotorCADPBCVisualizationBridge"
 )

for module in (
    pbc_boundary_module,
    pbc_candidate_module,
    pbc_case_module,
    pbc_module,
    pbc_bridge_module,
 ):
    importlib.reload(module)

MotorCADPBCVisualizationBridge = pbc_bridge_module.MotorCADPBCVisualizationBridge

PBC_VIS_CASE_INDICES = list(globals().get("DOE_CASE_INDICES", [0, 1])[:2])
if len(PBC_VIS_CASE_INDICES) < 2:
    PBC_VIS_CASE_INDICES = [0, 1]

PBC_VIS_SOURCE_TYPES = list(globals().get("DOE_SOURCE_FILE_TYPES", ["OnLoadTorque"]))
PBC_VIS_OUT_DIR = ROOT / "results" / "pbc_case_vis"

pbc_bridge = MotorCADPBCVisualizationBridge(ROOT)
pbc_cases, skipped_cases = pbc_bridge.collect_cases(
    case_indices=PBC_VIS_CASE_INDICES,
    source_file_types=PBC_VIS_SOURCE_TYPES,
    data_dir=ROOT / "doe_data",
    output_dir=PBC_VIS_OUT_DIR,
)

summary = {
    "case_indices": [int(case.case_idx) for case in pbc_cases],
    "source_file_types": sorted(
        {case.source_file_type for case in pbc_cases}
    ),
    "pairs": {
        str(case.case_idx): int(case.pbc_forward_index.shape[1])
        for case in pbc_cases
    },
    "groups": {
        str(case.case_idx): list(case.group_labels)
        for case in pbc_cases
    },
    "skipped_cases": skipped_cases,
}

print(json.dumps(summary, indent=2, ensure_ascii=False))
display(
    pbc_bridge.build_case_selector_widget(
        cases=pbc_cases,
        default_group="all",
        default_show_nodes=False,
        default_show_pbc_edges=False,
    )
)

## 9) SymMGN 3-case DOE Smoke Test

Full training 전에 3개 case로 파이프라인이 올바르게 작동하는지 확인합니다.

| 항목 | 값 |
|------|----|
| 모델 | SymMGN (anti-periodic PBC) |
| Case | 3개 (DOE index 0, 1, 2) |
| Source file type | OnLoadTorque |
| Max steps/case | 2 |
| Epochs | 2 |

In [ ]:
import json

DOE_CASE_INDICES = [0, 1, 2]
DOE_SOURCE_FILE_TYPES = ["OnLoadTorque"]
DOE_MAX_STEPS_PER_CASE = 2
DOE_EPOCHS = 2
DOE_BATCH_SIZE = 2
DOE_HIDDEN_DIM = 64
DOE_SEED = 42

SYM_CKPT_PATH = ROOT / "results" / "symm_mgn_doe_onloadtorque.pt"
SYM_TRAIN_LOG = "11_symm_train_doe.log"

train_cmd = " ".join(
    [
        "python -m phase1_static.train",
        "--input-format doe",
        "--data-dir doe_data",
        "--case-indices " + " ".join(str(idx) for idx in DOE_CASE_INDICES),
        "--source-file-types " + " ".join(DOE_SOURCE_FILE_TYPES),
        f"--max-steps-per-case {DOE_MAX_STEPS_PER_CASE}",
        f"--epochs {DOE_EPOCHS}",
        f"--batch-size {DOE_BATCH_SIZE}",
        f"--hidden-dim {DOE_HIDDEN_DIM}",
        f"--seed {DOE_SEED}",
        f"--ckpt-out results/{SYM_CKPT_PATH.name}",
    ]
)

cp_symm_train = run_docker(train_cmd, check=False)
save_log(SYM_TRAIN_LOG, cp_symm_train)

combined_train = (cp_symm_train.stdout or "") + "\n" + (cp_symm_train.stderr or "")
epoch_lines = [line for line in combined_train.splitlines() if "epoch=" in line]
pbc_skip_lines = [line for line in combined_train.splitlines() if "PBC boundary match skipped" in line]

print("=== SymMGN DOE 학습 (Docker) ===")
print(
    json.dumps(
        {
            "returncode": cp_symm_train.returncode,
            "case_indices": DOE_CASE_INDICES,
            "source_file_types": DOE_SOURCE_FILE_TYPES,
            "last_epoch": epoch_lines[-1] if epoch_lines else "",
            "pbc_skip_count": len(pbc_skip_lines),
            "ckpt_saved": SYM_CKPT_PATH.exists(),
            "ckpt_size_kb": round(SYM_CKPT_PATH.stat().st_size / 1024, 1) if SYM_CKPT_PATH.exists() else 0,
            "log_file": str(LOG_DIR / SYM_TRAIN_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

if cp_symm_train.returncode != 0:
    print("=== train stderr tail ===")
    print("\n".join(combined_train.splitlines()[-40:]))
    raise RuntimeError("DOE 실데이터 SymMGN 학습 실패. 11_symm_train_doe.log를 확인하세요.")

## 10) Smoke Test 추론 + 시각화

3-case smoke 모델로 case 0의 첫 step을 추론하고 GT vs Pred 시각화를 확인합니다.

In [ ]:
import json

DOE_SOURCE_FILE_TYPES = globals().get("DOE_SOURCE_FILE_TYPES", ["OnLoadTorque"])
SYM_CKPT_PATH = globals().get("SYM_CKPT_PATH", ROOT / "results" / "symm_mgn_doe_onloadtorque.pt")
DOE_INFER_CASE_IDX = globals().get("DOE_CASE_INDICES", [0])[0]
DOE_INFER_MAX_STEPS = 1
SYM_INFER_NPZ_PATH = ROOT / "results" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_onloadtorque_step1.npz"
SYM_INFER_LOG = "12_symm_infer_doe.log"

infer_cmd = " ".join(
    [
        "python infer_phase1_pbc.py",
        f"--ckpt /workspace/app/results/{SYM_CKPT_PATH.name}",
        "--data-dir /workspace/app/doe_data",
        f"--case-idx {DOE_INFER_CASE_IDX}",
        "--source-file-types " + " ".join(DOE_SOURCE_FILE_TYPES),
        f"--max-steps {DOE_INFER_MAX_STEPS}",
        "--batch-size 1",
        f"--out /workspace/app/results/{SYM_INFER_NPZ_PATH.name}",
    ]
)

cp_symm_infer = run_docker(infer_cmd, check=False)
save_log(SYM_INFER_LOG, cp_symm_infer)

combined_infer = (cp_symm_infer.stdout or "") + "\n" + (cp_symm_infer.stderr or "")
metric_lines = [line for line in combined_infer.splitlines() if "RMSE=" in line]

print("=== SymMGN DOE 추론 (Docker) ===")
print(
    json.dumps(
        {
            "returncode": cp_symm_infer.returncode,
            "case_idx": DOE_INFER_CASE_IDX,
            "source_file_types": DOE_SOURCE_FILE_TYPES,
            "max_steps": DOE_INFER_MAX_STEPS,
            "infer_npz_exists": SYM_INFER_NPZ_PATH.exists(),
            "infer_npz_size_kb": round(SYM_INFER_NPZ_PATH.stat().st_size / 1024, 1) if SYM_INFER_NPZ_PATH.exists() else 0,
            "metric_lines": metric_lines[-5:],
            "log_file": str(LOG_DIR / SYM_INFER_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

if cp_symm_infer.returncode != 0:
    print("=== infer stderr tail ===")
    print("\n".join(combined_infer.splitlines()[-40:]))
    raise RuntimeError("DOE 실데이터 SymMGN 추론 실패. 12_symm_infer_doe.log를 확인하세요.")

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

DOE_INFER_CASE_IDX = globals().get("DOE_INFER_CASE_IDX", 0)
SYM_INFER_NPZ_PATH = globals().get(
    "SYM_INFER_NPZ_PATH",
    ROOT / "results" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_onloadtorque_step1.npz",
)
SYM_VIS_PNG_PATH = ROOT / "logs" / f"symm_mgn_case{DOE_INFER_CASE_IDX:04d}_gt_vs_pred.png"

if not SYM_INFER_NPZ_PATH.exists():
    raise FileNotFoundError(f"추론 NPZ가 없습니다: {SYM_INFER_NPZ_PATH}")

arr = np.load(SYM_INFER_NPZ_PATH, allow_pickle=True)
pos_x = arr["pos_x"]
pos_y = arr["pos_y"]
channels = ["bx", "by", "a", "je"]
labels = ["Bx", "By", "A", "Je"]

fig, axes = plt.subplots(2, len(channels), figsize=(20, 7))
fig.suptitle(
    f"SymMGN DOE case {DOE_INFER_CASE_IDX:04d} - OnLoadTorque GT vs Prediction",
    fontsize=12,
)

for col_idx, (channel_name, label) in enumerate(zip(channels, labels)):
    gt_values = arr[f"gt_{channel_name}"]
    pred_values = arr[f"pred_{channel_name}"]
    vmin = float(min(gt_values.min(), pred_values.min()))
    vmax = float(max(gt_values.max(), pred_values.max()))

    gt_plot = axes[0, col_idx].scatter(
        pos_x,
        pos_y,
        c=gt_values,
        cmap="RdBu_r",
        s=10,
        vmin=vmin,
        vmax=vmax,
    )
    axes[0, col_idx].set_title(f"GT {label}")
    axes[0, col_idx].set_aspect("equal")
    axes[0, col_idx].set_xticks([])
    axes[0, col_idx].set_yticks([])
    plt.colorbar(gt_plot, ax=axes[0, col_idx], fraction=0.04)

    pred_plot = axes[1, col_idx].scatter(
        pos_x,
        pos_y,
        c=pred_values,
        cmap="RdBu_r",
        s=10,
        vmin=vmin,
        vmax=vmax,
    )
    axes[1, col_idx].set_title(f"Pred {label}")
    axes[1, col_idx].set_aspect("equal")
    axes[1, col_idx].set_xticks([])
    axes[1, col_idx].set_yticks([])
    plt.colorbar(pred_plot, ax=axes[1, col_idx], fraction=0.04)

plt.tight_layout()
SYM_VIS_PNG_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(SYM_VIS_PNG_PATH, dpi=110, bbox_inches="tight")
plt.show()
plt.close()

meta = json.loads(arr["meta"].item()) if "meta" in arr.files else {}
metrics = json.loads(arr["metrics"].item()) if "metrics" in arr.files else {}

print(
    json.dumps(
        {
            "saved_png": str(SYM_VIS_PNG_PATH),
            "png_size_kb": round(SYM_VIS_PNG_PATH.stat().st_size / 1024, 1),
            "meta": meta,
            "metrics": metrics,
        },
        indent=2,
        ensure_ascii=False,
    )
)

# Section: 학습 데이터 품질 시각화

학습에 투입되는 샘플의 물리 필드와 경계 조건을 눈으로 확인합니다.

**포함 내용:**
- **Plot 1** – 4채널 필드 맵 (Bx, By, A, Je) + j_raw 비교  
- **Plot 2** – 메쉬 경계 조건 (Dirichlet outer BC, interior edges 밀도)  
- **Plot 3** – 주기 경계 조건 (PBC): stationary(고정자) / moving(회전자) / slide band 구분  

**핵심 region 정보 (case_0000)**

| code | name | 분류 |
|---|---|---|
| 1, 55-59 | Stator, Stator_N | 고정자 철심 |
| 2-7, 31-54 | ArmatureSlot* | 고정자 권선 슬롯 |
| 10 | Rotor | 회전자 철심 |
| 60,68,76,84 | Magnet* | 영구자석 |
| 93-96 | a1-a4 | Slide Band (air gap) |
| 109-112 | Turn_*_1 | 코일 권선 |

`moving_reg_codes` = [10, 13, 14, 17, 18, 19, 60, 68, 76, 84, 94, 95, 96] → 회전자 패밀리


In [ ]:
# ── Cell VIZ-0: 샘플 로드 + region 맵 준비 ──────────────────────────────────
# numpy + h5py only (torch 불필요 — 컨테이너 전용)
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, r"D:\KDH\NvidiaNemo")

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import h5py, json, pathlib

# ── 파라미터 ──────────────────────────────────────────────────────────────────
DOE_DATA_DIR = r"D:\KDH\NvidiaNemo\doe_data"
CASE_IDX = 17          # 시각화할 케이스 인덱스
STEP_IDX = 31        # H5 step 인덱스 (0 = 첫 번째 step)
SOURCE_TYPE = "OnLoadTorque"

# ── H5 경로 ───────────────────────────────────────────────────────────────────
manifest = json.loads(pathlib.Path(f"{DOE_DATA_DIR}/doe_manifest.json").read_text())
case = next(c for c in manifest["cases"] if c["index"] == CASE_IDX)
h5_path = next(
    p for p in case["h5_paths"]
    if SOURCE_TYPE.lower() in pathlib.Path(p).name.lower().replace("\\", "/")
)
h5_local = pathlib.Path(
    str(h5_path)
    .replace("D:\\KDH\\Sim_4SolverX\\DOE_TrainingData", DOE_DATA_DIR)
    .replace("/", "\\")
)
if not h5_local.exists():
    h5_local = pathlib.Path(DOE_DATA_DIR) / f"case_{CASE_IDX:04d}" / "postproc" / pathlib.Path(h5_path).name
print(f"H5 파일: {h5_local}")
assert h5_local.exists(), f"H5 파일을 찾을 수 없습니다: {h5_local}"

# ── H5 파싱 ──────────────────────────────────────────────────────────────────
with h5py.File(h5_local, "r") as f:
    steps        = np.asarray(f["steps"][:] if "steps" in f else [int(f["step"][()])], dtype=np.int32)
    node_id      = np.asarray(f["mesh/node_id"][:], dtype=np.int32)
    node_x0      = np.asarray(f["mesh/node_x_mm"][:], dtype=np.float64)
    node_y0      = np.asarray(f["mesh/node_y_mm"][:], dtype=np.float64)
    raw_n1       = np.asarray(f["mesh/node_1"][:], dtype=np.int32)
    raw_n2       = np.asarray(f["mesh/node_2"][:], dtype=np.int32)
    raw_n3       = np.asarray(f["mesh/node_3"][:], dtype=np.int32)
    reg_code_raw = np.asarray(f["mesh/reg_code"][:], dtype=np.int32)
    moving_reg_codes = (
        np.asarray(f["mesh/moving_reg_codes"][:], dtype=np.int32)
        if "mesh/moving_reg_codes" in f else np.array([], dtype=np.int32)
    )
    moving_idx = (
        np.asarray(f["mesh/moving_node_indices"][:], dtype=np.int32)
        if "mesh/moving_node_indices" in f else None
    )
    x_bsm = np.asarray(f["mesh/node_x_mm_by_step_moving"][:], dtype=np.float64) if "mesh/node_x_mm_by_step_moving" in f else None
    y_bsm = np.asarray(f["mesh/node_y_mm_by_step_moving"][:], dtype=np.float64) if "mesh/node_y_mm_by_step_moving" in f else None
    x_bs  = np.asarray(f["mesh/node_x_mm_by_step"][:], dtype=np.float64) if "mesh/node_x_mm_by_step" in f else None
    y_bs  = np.asarray(f["mesh/node_y_mm_by_step"][:], dtype=np.float64) if "mesh/node_y_mm_by_step" in f else None
    reg_codes_meta = np.asarray(f["regions/reg_code"][:], dtype=np.int32)
    reg_names_meta = [n.decode("utf-8") if isinstance(n, bytes) else str(n) for n in f["regions/name"][:]]
    REGION_NAME    = {int(c): str(n) for c, n in zip(reg_codes_meta, reg_names_meta)}
    bx_all = np.asarray(f["fields/bx"][:], dtype=np.float32)
    by_all = np.asarray(f["fields/by"][:], dtype=np.float32)
    a_all  = np.asarray(f["fields/a"][:],  dtype=np.float32) if "fields/a"  in f else np.zeros_like(bx_all)
    j_all  = np.asarray(f["fields/j"][:],  dtype=np.float32) if "fields/j"  in f else np.zeros_like(bx_all)
    je_all = np.asarray(f["fields/je"][:], dtype=np.float32) if "fields/je" in f else np.zeros_like(bx_all)

# ── node_id → 0-based index LUT ──────────────────────────────────────────────
sorted_ids = np.sort(node_id)
max_nid    = int(node_id.max()) + 1
lut        = np.full(max_nid, -1, dtype=np.int64)
for i, nid in enumerate(sorted_ids):
    lut[nid] = i
sort_order = np.argsort(node_id)
N = len(sorted_ids)

# ── triangle 인덱스 변환 ──────────────────────────────────────────────────────
m = min(len(raw_n1), len(raw_n2), len(raw_n3), len(reg_code_raw))
i1 = lut[np.clip(raw_n1[:m], 0, max_nid - 1)]
i2 = lut[np.clip(raw_n2[:m], 0, max_nid - 1)]
i3 = lut[np.clip(raw_n3[:m], 0, max_nid - 1)]
valid_elem   = (i1 >= 0) & (i2 >= 0) & (i3 >= 0)
i1v, i2v, i3v = i1[valid_elem], i2[valid_elem], i3[valid_elem]
reg_code_elem = reg_code_raw[:m][valid_elem].astype(np.int32)
triangles_np  = np.stack([i1v, i2v, i3v], axis=1)

# ── step별 노드 좌표 ──────────────────────────────────────────────────────────
x = node_x0.copy()
y = node_y0.copy()
if x_bs is not None and y_bs is not None:
    xs, ys = x_bs[STEP_IDX], y_bs[STEP_IDX]
    ok = np.isfinite(xs) & np.isfinite(ys)
    x[ok], y[ok] = xs[ok], ys[ok]
if x_bsm is not None and y_bsm is not None and moving_idx is not None and moving_idx.size > 0:
    n_nodes     = node_id.size
    mov_mask    = (moving_idx >= 0) & (moving_idx < n_nodes)
    xs_mov, ys_mov = x_bsm[STEP_IDX], y_bsm[STEP_IDX]
    n_mov       = min(len(xs_mov), mov_mask.sum())
    valid_mi    = np.where(mov_mask)[0][:n_mov]
    node_indices = moving_idx[valid_mi]
    xv, yv      = xs_mov[valid_mi], ys_mov[valid_mi]
    ok_fin      = np.isfinite(xv) & np.isfinite(yv)
    x[node_indices[ok_fin]] = xv[ok_fin]
    y[node_indices[ok_fin]] = yv[ok_fin]
node_x = x[sort_order].astype(np.float32)
node_y = y[sort_order].astype(np.float32)

# ── step별 element 필드 ───────────────────────────────────────────────────────
bx_mat = bx_all.reshape(1, -1) if bx_all.ndim == 1 else bx_all
by_mat = by_all.reshape(1, -1) if by_all.ndim == 1 else by_all
a_mat  =  a_all.reshape(1, -1) if  a_all.ndim == 1 else  a_all
j_mat  =  j_all.reshape(1, -1) if  j_all.ndim == 1 else  j_all
je_mat = je_all.reshape(1, -1) if je_all.ndim == 1 else je_all
bx_step = bx_mat[STEP_IDX][valid_elem]
by_step = by_mat[STEP_IDX][valid_elem]
a_step  =  a_mat[STEP_IDX][valid_elem]
j_step  =  j_mat[STEP_IDX][valid_elem]
je_step = je_mat[STEP_IDX][valid_elem]

def elem_to_node_avg(field_elem, n_nodes, tri):
    acc = np.zeros(n_nodes, dtype=np.float64)
    cnt = np.zeros(n_nodes, dtype=np.int32)
    for ch in range(3):
        np.add.at(acc, tri[:, ch], field_elem)
        np.add.at(cnt, tri[:, ch], 1)
    return (acc / np.maximum(cnt, 1)).astype(np.float32)

bx_node  = elem_to_node_avg(bx_step, N, triangles_np)
by_node  = elem_to_node_avg(by_step, N, triangles_np)
a_node   = elem_to_node_avg(a_step,  N, triangles_np)
j_node   = elem_to_node_avg(j_step,  N, triangles_np)
je_node  = elem_to_node_avg(je_step, N, triangles_np)

node_reg = np.zeros(N, dtype=np.int32)
for e_idx in range(len(triangles_np)):
    for n in triangles_np[e_idx]:
        if node_reg[n] == 0:
            node_reg[n] = reg_code_elem[e_idx]

MOVING_SET      = set(moving_reg_codes.tolist())
SLOT_CODES      = {c for c, n in REGION_NAME.items() if "armature" in n.lower() or "turn" in n.lower() or "impreg" in n.lower()}
SLIDEBAND_CODES = {93, 94, 95, 96}

triang = mtri.Triangulation(node_x, node_y, triangles_np)
print(f"로드 완료 — step={STEP_IDX}/{len(steps)}, nodes={N}, elements={len(triangles_np)}")
print(f"  |Bx| max={bx_node.max():.4f} T,  |By| max={by_node.max():.4f} T")
print(f"  |A|  max={a_node.max():.4e} Wb/m,  |Je| max={je_node.max():.4f} A/m²")
print(f"  |j_raw(node)| max={j_node.max():.4f} A/m²")
print(f"  moving region codes: {sorted(MOVING_SET)}")


In [ ]:
# ── Cell VIZ-1: 4채널 필드 맵 (Bx, By, A, Je) + j_raw ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    f"Field Maps — Case {CASE_IDX:04d}, {SOURCE_TYPE}, step={STEP_IDX}",
    fontsize=14, fontweight="bold"
)

FIELDS = [
    (bx_node,  "Bx [T]",     "RdBu_r"),
    (by_node,  "By [T]",     "RdBu_r"),
    (a_node,   "A [Wb/m]",   "plasma"),
    (je_node,  "Je [A/m²]",  "hot"),
    (j_node,   "J_raw [A/m²] (FEM ref)", "hot"),
    (je_node - j_node, "Je − J_raw [A/m²]", "coolwarm"),
]

for ax, (field, label, cmap) in zip(axes.flat, FIELDS):
    vmax = np.abs(field).max() or 1.0
    vmin = -vmax if cmap in ("RdBu_r", "coolwarm") else 0.0
    tc = ax.tripcolor(triang, field, cmap=cmap, vmin=vmin, vmax=vmax, shading="gouraud")
    fig.colorbar(tc, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(label, fontsize=11)
    ax.set_aspect("equal")
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(f"viz_fields_case{CASE_IDX:04d}_step{STEP_IDX}.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ 저장: viz_fields_case***.png")


In [ ]:
# ── Cell VIZ-2: 경계 조건 시각화 ────────────────────────────────────────────
# (a) Region 컬러맵  (b) Dirichlet A=0 외부 경계  (c) Interior edge 밀도
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle(f"Mesh & Boundary Conditions — Case {CASE_IDX:04d}, step={STEP_IDX}", fontsize=13, fontweight="bold")

ax = axes[0]
ax.set_title("Region 분류")

REGION_COLOR = {
    "stator":    "#4a90d9",
    "slot":      "#f5a623",
    "rotor":     "#d0021b",
    "magnet":    "#7ed321",
    "slideband": "#9b59b6",
    "shaft":     "#8b572a",
    "air":       "#cccccc",
}

def classify_region(code):
    name = REGION_NAME.get(code, "").lower()
    if "stator" in name and "air" not in name and "wedge" not in name:
        return "stator"
    if "armature" in name or "turn" in name or "impreg" in name:
        return "slot"
    if "rotor" in name or "pocket" in name:
        return "rotor"
    if "magnet" in name:
        return "magnet"
    if code in SLIDEBAND_CODES:
        return "slideband"
    if "shaft" in name:
        return "shaft"
    return "air"

elem_class = np.array([classify_region(c) for c in reg_code_elem])
CLASS_ORDER = ["stator", "slot", "rotor", "magnet", "slideband", "shaft", "air"]
class_to_idx = {c: i for i, c in enumerate(CLASS_ORDER)}
elem_color_idx = np.array([class_to_idx[c] for c in elem_class])

cmap_reg = matplotlib.colors.ListedColormap([REGION_COLOR[c] for c in CLASS_ORDER])
tc = ax.tripcolor(triang, elem_color_idx, cmap=cmap_reg, vmin=-0.5, vmax=len(CLASS_ORDER)-0.5)
legend_patches = [mpatches.Patch(facecolor=REGION_COLOR[c], label=c) for c in CLASS_ORDER]
ax.legend(handles=legend_patches, loc="upper right", fontsize=7, framealpha=0.8)
ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")

ax = axes[1]
ax.set_title("외부 경계 A=0 (Dirichlet)")
ax.tripcolor(triang, a_node, cmap="plasma", shading="gouraud", alpha=0.6)

from collections import Counter
edge_count = Counter()
for n1, n2, n3 in triangles_np.tolist():
    for u, v in [(n1,n2),(n2,n3),(n3,n1)]:
        edge_count[tuple(sorted([u,v]))] += 1
outer_edges = [e for e, cnt in edge_count.items() if cnt == 1]
segs_outer = [[[node_x[u], node_y[u]], [node_x[v], node_y[v]]] for u,v in outer_edges]
lc_outer = LineCollection(segs_outer, colors="red", linewidths=1.5, label="Dirichlet A=0 outer")
ax.add_collection(lc_outer)
ax.legend(fontsize=8, loc="upper right")
ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
ax.autoscale_view()

ax = axes[2]
ax.set_title("Interior Edges (학습 그래프 구조)")
ax.set_facecolor("#111111")
inner_edges = [e for e, cnt in edge_count.items() if cnt == 2]
rng = np.random.default_rng(42)
if len(inner_edges) > 50000:
    inner_edges = [inner_edges[i] for i in rng.choice(len(inner_edges), 50000, replace=False)]
segs_inner = [[[node_x[u], node_y[u]], [node_x[v], node_y[v]]] for u,v in inner_edges]
lc_inner = LineCollection(segs_inner, colors="#88aaff", linewidths=0.2, alpha=0.4)
ax.add_collection(lc_inner)
ax.scatter(node_x, node_y, s=0.3, c="white", alpha=0.3)
ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]")
ax.autoscale_view()

plt.tight_layout()
plt.savefig(f"viz_bc_case{CASE_IDX:04d}_step{STEP_IDX}.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"→ 외부 boundary edge 수: {len(outer_edges)}, interior edge 수: {len(inner_edges)}")


In [ ]:
# ── Cell VIZ-3: 주기 경계 조건 (PBC) 시각화 (선 연결 없이 pair 색상으로 표시) ──
import sys
sys.path.insert(0, r"D:\KDH\NvidiaNemo")

from phase1_static.pbc_boundary import extract_periodic_boundary_groups_from_mesh

pos_xy = np.stack([node_x, node_y], axis=1).astype(np.float64)
tri_arr = triangles_np.astype(np.int32)
reg_arr = reg_code_elem.astype(np.int32)

group_specs, group_diag, error_code = extract_periodic_boundary_groups_from_mesh(
    pos_xy,
    tri_arr,
    reg_arr,
    moving_reg_codes=moving_reg_codes,
    region_name_by_code=REGION_NAME,
    rotation_deg=-45.0,
)
print(f"PBC group 추출: labels={group_diag.get('group_labels')}, error={error_code}")

sb_mask = np.isin(node_reg, list(SLIDEBAND_CODES))
sb_nodes_x = node_x[sb_mask]
sb_nodes_y = node_y[sb_mask]

origin_xy = np.asarray(group_diag.get("origin_xy", (0.0, 0.0)), dtype=np.float64)
ox, oy = float(origin_xy[0]), float(origin_xy[1])

GROUP_STYLE = {
    "stationary": {"edge": "#27ae60", "label": "고정자 PBC"},
    "moving": {"edge": "#c0392b", "label": "회전자 PBC"},
    "all": {"edge": "#2980b9", "label": "전체 PBC"},
}

# 그룹별 pair를 동일 색으로 표시하기 위한 컬러맵
pair_cmap = plt.get_cmap("tab20")

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
fig.suptitle(
    f"PBC 시각화(연결선 없음) — Case {CASE_IDX:04d}, step={STEP_IDX}",
    fontsize=13,
    fontweight="bold",
)


def draw_base(ax, alpha_field=0.35):
    ax.tripcolor(triang, a_node, cmap="plasma", shading="gouraud", alpha=alpha_field)
    if "segs_outer" in globals() and len(segs_outer) > 0:
        lc = LineCollection(segs_outer, colors="white", linewidths=0.6, alpha=0.35)
        ax.add_collection(lc)
    ax.set_aspect("equal")
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
    ax.autoscale_view()


# (a) XY overview: master/slave를 동일 pair 색으로 점만 표시
ax = axes[0]
ax.set_title("XY Overview (● master / ▲ slave, 같은 색=linked pair)")
draw_base(ax)

for grp_label, master_idx, slave_idx in group_specs:
    style = GROUP_STYLE.get(grp_label, {"edge": "#7f8c8d", "label": grp_label})

    m_xy_all = pos_xy[np.asarray(master_idx, dtype=np.int64)]
    s_xy_all = pos_xy[np.asarray(slave_idx, dtype=np.int64)]
    n_pair = min(len(m_xy_all), len(s_xy_all))

    if n_pair > 0:
        colors = [pair_cmap(i % 20) for i in range(n_pair)]
        m_xy = m_xy_all[:n_pair]
        s_xy = s_xy_all[:n_pair]

        ax.scatter(
            m_xy[:, 0],
            m_xy[:, 1],
            c=colors,
            s=18,
            marker="o",
            edgecolors=style["edge"],
            linewidths=0.5,
            alpha=0.95,
            label=f"{style['label']} master",
        )
        ax.scatter(
            s_xy[:, 0],
            s_xy[:, 1],
            c=colors,
            s=20,
            marker="^",
            edgecolors=style["edge"],
            linewidths=0.5,
            alpha=0.95,
            label=f"{style['label']} slave",
        )

    # pair로 못 맞춘 잔여 점은 회색으로 표시
    if len(m_xy_all) > n_pair:
        rem = m_xy_all[n_pair:]
        ax.scatter(rem[:, 0], rem[:, 1], c="lightgray", s=12, marker="o", alpha=0.5)
    if len(s_xy_all) > n_pair:
        rem = s_xy_all[n_pair:]
        ax.scatter(rem[:, 0], rem[:, 1], c="lightgray", s=12, marker="^", alpha=0.5)

if sb_mask.sum() > 0:
    ax.scatter(sb_nodes_x, sb_nodes_y, s=4, c="#9b59b6", alpha=0.25, label="Slide Band")

ax.scatter([ox], [oy], s=70, c="yellow", marker="*", edgecolors="black", linewidths=0.4, zorder=10)
ax.legend(fontsize=7, loc="upper right", framealpha=0.9)


# (b) Polar view: (theta, r)에서 같은 pair 색 표시
ax = axes[1]
ax.set_title("Polar View (theta-r, 같은 색=linked pair)")
for grp_label, master_idx, slave_idx in group_specs:
    style = GROUP_STYLE.get(grp_label, {"edge": "#7f8c8d", "label": grp_label})

    m_xy_all = pos_xy[np.asarray(master_idx, dtype=np.int64)]
    s_xy_all = pos_xy[np.asarray(slave_idx, dtype=np.int64)]
    n_pair = min(len(m_xy_all), len(s_xy_all))
    if n_pair == 0:
        continue

    colors = [pair_cmap(i % 20) for i in range(n_pair)]
    m_xy = m_xy_all[:n_pair]
    s_xy = s_xy_all[:n_pair]

    m_rel = m_xy - origin_xy[None, :]
    s_rel = s_xy - origin_xy[None, :]

    m_th = np.degrees(np.arctan2(m_rel[:, 1], m_rel[:, 0]))
    s_th = np.degrees(np.arctan2(s_rel[:, 1], s_rel[:, 0]))
    m_r = np.linalg.norm(m_rel, axis=1)
    s_r = np.linalg.norm(s_rel, axis=1)

    ax.scatter(m_th, m_r, c=colors, s=18, marker="o", edgecolors=style["edge"], linewidths=0.4, alpha=0.95)
    ax.scatter(s_th, s_r, c=colors, s=20, marker="^", edgecolors=style["edge"], linewidths=0.4, alpha=0.95)

ax.set_xlabel("theta [deg]")
ax.set_ylabel("radius [mm]")
ax.grid(alpha=0.3)


# (c) Slide band zoom: 연결선 없이 점만
ax = axes[2]
ax.set_title("Slide Band Zoom (점 기반 pair 표현)")
draw_base(ax, alpha_field=0.28)

if sb_mask.sum() > 0:
    sb_elem_mask = np.isin(reg_code_elem, list(SLIDEBAND_CODES))
    if np.any(sb_elem_mask):
        sb_triang = mtri.Triangulation(node_x, node_y, tri_arr[sb_elem_mask])
        ax.triplot(sb_triang, color="#9b59b6", linewidth=0.45, alpha=0.7)
    ax.scatter(sb_nodes_x, sb_nodes_y, s=6, c="#9b59b6", zorder=4, alpha=0.5)

for grp_label, master_idx, slave_idx in group_specs:
    style = GROUP_STYLE.get(grp_label, {"edge": "#7f8c8d", "label": grp_label})

    m_xy_all = pos_xy[np.asarray(master_idx, dtype=np.int64)]
    s_xy_all = pos_xy[np.asarray(slave_idx, dtype=np.int64)]
    n_pair = min(len(m_xy_all), len(s_xy_all))
    if n_pair == 0:
        continue

    colors = [pair_cmap(i % 20) for i in range(n_pair)]
    m_xy = m_xy_all[:n_pair]
    s_xy = s_xy_all[:n_pair]

    ax.scatter(m_xy[:, 0], m_xy[:, 1], c=colors, s=24, marker="o", edgecolors=style["edge"], linewidths=0.5, zorder=6)
    ax.scatter(s_xy[:, 0], s_xy[:, 1], c=colors, s=26, marker="^", edgecolors=style["edge"], linewidths=0.5, zorder=6)

if sb_mask.sum() > 0:
    xc = float(np.mean(sb_nodes_x))
    yc = float(np.mean(sb_nodes_y))
    r_sb = float(max(np.abs(sb_nodes_x - xc).max(), np.abs(sb_nodes_y - yc).max()) * 1.5 + 2.0)
    ax.set_xlim(xc - r_sb, xc + r_sb)
    ax.set_ylim(yc - r_sb, yc + r_sb)

legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markeredgecolor="black", markersize=6, label="master"),
    plt.Line2D([0], [0], marker="^", color="w", markerfacecolor="gray", markeredgecolor="black", markersize=6, label="slave"),
    mpatches.Patch(facecolor="#9b59b6", label="Slide Band"),
]
ax.legend(handles=legend_handles, fontsize=7, loc="upper right", framealpha=0.9)

plt.tight_layout()
out_path = f"viz_pbc_case{CASE_IDX:04d}_step{STEP_IDX}_paircolor.png"
plt.savefig(out_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"→ 저장: {out_path}")
print(f"그룹 라벨: {group_diag.get('group_labels')}, 그룹 수: {len(group_specs)}")

In [ ]:
# ── Cell VIZ-4: 다중 케이스 PBC match_ratio 배치 검증 ──────────────────────
# 모든 케이스의 pbc_match_ratio를 한 번에 확인하는 요약 플롯
# (이 셀은 시간이 오래 걸릴 수 있습니다 — 먼저 VIZ-0~3을 실행한 뒤 실행)

from phase1_static.motor_dataset import build_samples_from_doe_manifest
import pandas as pd

print("DOE 전체 케이스 PBC match_ratio 로드 중 (max_steps_per_case=1)...")
all_samples = build_samples_from_doe_manifest(
    DOE_DATA_DIR,
    max_steps_per_case=1,
    source_file_types=["OnLoadTorque"],
)

rows = []
for s in all_samples:
    rows.append({
        "case_idx":       s["case_idx"],
        "step_index":     s["step_index"],
        "pbc_match_ratio": float(s.get("pbc_match_ratio", 0.0)),
        "pbc_error_code":  str(s.get("pbc_error_code", "")),
        "n_nodes":         s["pos"].shape[0],
        "n_pbc_edges":     s["pbc_edge_index"].shape[1],
    })

df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PBC Quality Across DOE Cases (OnLoadTorque, step=0)", fontsize=12, fontweight="bold")

# (a) match_ratio histogram
ax = axes[0]
ax.hist(df["pbc_match_ratio"], bins=20, color="#3498db", edgecolor="white")
ax.axvline(1.0, color="green", ls="--", lw=2, label="ratio=1.0 (완벽 매칭)")
ax.axvline(0.9, color="orange", ls="--", lw=1.5, label="ratio=0.9 (경계)")
ax.set_xlabel("PBC match_ratio")
ax.set_ylabel("케이스 수")
ax.set_title("PBC 매칭 품질 분포")
ax.legend()

# (b) scatter: n_nodes vs n_pbc_edges, colored by match_ratio
ax = axes[1]
sc = ax.scatter(
    df["n_nodes"], df["n_pbc_edges"],
    c=df["pbc_match_ratio"], cmap="RdYlGn", vmin=0.8, vmax=1.0,
    s=40, edgecolors="gray", linewidths=0.3
)
plt.colorbar(sc, ax=ax, label="pbc_match_ratio")
ax.set_xlabel("노드 수")
ax.set_ylabel("PBC edge 수")
ax.set_title("노드 수 vs PBC edge 수")

# 완벽하지 않은 케이스 표시
bad = df[df["pbc_match_ratio"] < 1.0]
if len(bad) > 0:
    ax.scatter(bad["n_nodes"], bad["n_pbc_edges"], c="red", s=80, marker="x",
               linewidths=2, zorder=5, label="match_ratio < 1.0")
    ax.legend()
    print(f"\n⚠ PBC 불완전 케이스: {len(bad)}개")
    print(bad[["case_idx","step_index","pbc_match_ratio","pbc_error_code"]].to_string(index=False))
else:
    print(f"\n✅ 전체 {len(df)}개 케이스 모두 PBC match_ratio=1.0")

plt.tight_layout()
plt.savefig("viz_pbc_quality_all.png", dpi=110, bbox_inches="tight")
plt.show()
print(f"\n요약:\n{df[['pbc_match_ratio','n_nodes','n_pbc_edges']].describe().round(2)}")


# 학습 수행

## 11) Full 40-case DOE 학습

DOE manifest의 전체 40개 케이스를 사용해 SymMGN을 본격 학습합니다.

| 항목 | 값 |
|------|----|
| 모델 | SymMGN (anti-periodic PBC) |
| Case | **40개** (DOE 전체) |
| Source file type | OnLoadTorque |
| Max steps/case | 1 (static 첫 step) |
| Epochs | 50 |
| Hidden dim | 128 |
| Batch size | 4 |

> ⚠ GPU 시간이 상당히 소요됩니다. 진행 상황은 로그 파일에서 확인하세요.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import shutil

ROOT = globals().get("ROOT", Path.cwd())
LOG_DIR = globals().get("LOG_DIR", ROOT / "logs" / "tutorial")
LOG_DIR.mkdir(parents=True, exist_ok=True)

FULL_CASE_INDICES = list(range(40))
FULL_SOURCE_FILE_TYPES = ["OnLoadTorque"]
FULL_MAX_STEPS_PER_CASE = 10
FULL_EPOCHS = 100
FULL_BATCH_SIZE = 4
FULL_HIDDEN_DIM = 128
FULL_SEED = 42

FULL_CKPT_PATH = ROOT / "results" / "symm_mgn_doe_full40_onloadtorque.pt"
FULL_TRAIN_LOG = "20_full40_train.log"

# ckpt가 이미 있으면 기본적으로 재학습 생략
SKIP_IF_CKPT_EXISTS = False

# 재학습할 경우, 기존 ckpt 백업
if (not SKIP_IF_CKPT_EXISTS) and FULL_CKPT_PATH.exists():
    ckpt_backup_dir = ROOT / "results" / "backups" / "ckpt"
    ckpt_backup_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    dst = ckpt_backup_dir / f"{FULL_CKPT_PATH.stem}_{ts}{FULL_CKPT_PATH.suffix}"
    shutil.copy2(FULL_CKPT_PATH, dst)
    print(f"[backup] ckpt -> {dst}")

if SKIP_IF_CKPT_EXISTS and FULL_CKPT_PATH.exists():
    print(f"[skip] 기존 체크포인트 사용: {FULL_CKPT_PATH}")
    print("재학습이 필요하면 SKIP_IF_CKPT_EXISTS = False 로 바꿔서 다시 실행하세요.")
else:
    # 학습 재실행 전 기존 full40 infer npz 백업
    if "backup_npz_dir" in globals():
        backup_npz_dir(ROOT / "results" / "full40_infer", label="before_full40_retrain")

    train_cmd = " ".join(
        [
            "python -m phase1_static.train",
            "--input-format doe",
            "--data-dir doe_data",
            "--case-indices " + " ".join(str(i) for i in FULL_CASE_INDICES),
            "--source-file-types " + " ".join(FULL_SOURCE_FILE_TYPES),
            f"--max-steps-per-case {FULL_MAX_STEPS_PER_CASE}",
            f"--epochs {FULL_EPOCHS}",
            f"--batch-size {FULL_BATCH_SIZE}",
            f"--hidden-dim {FULL_HIDDEN_DIM}",
            f"--seed {FULL_SEED}",
            f"--ckpt-out results/{FULL_CKPT_PATH.name}",
        ]
    )

    cp_full_train = run_docker(train_cmd, check=False)
    save_log(FULL_TRAIN_LOG, cp_full_train)

    combined = (cp_full_train.stdout or "") + "\n" + (cp_full_train.stderr or "")
    epoch_lines = [line for line in combined.splitlines() if "epoch=" in line]

    print(
        json.dumps(
            {
                "returncode": cp_full_train.returncode,
                "case_count": len(FULL_CASE_INDICES),
                "source_file_types": FULL_SOURCE_FILE_TYPES,
                "max_steps_per_case": FULL_MAX_STEPS_PER_CASE,
                "epochs": FULL_EPOCHS,
                "last_epoch": epoch_lines[-1] if epoch_lines else "",
                "ckpt_saved": FULL_CKPT_PATH.exists(),
                "ckpt_size_kb": round(FULL_CKPT_PATH.stat().st_size / 1024, 1) if FULL_CKPT_PATH.exists() else 0,
                "log_file": str(LOG_DIR / FULL_TRAIN_LOG),
            },
            indent=2,
            ensure_ascii=False,
        )
    )

    if cp_full_train.returncode != 0:
        print("=== train stderr tail ===")
        print("\n".join(combined.splitlines()[-40:]))
        raise RuntimeError(f"Full 40-case 학습 실패. {FULL_TRAIN_LOG} 를 확인하세요.")


## 12) Full 40-case 추론

학습된 Full 40-case 모델로 전체 케이스를 순회하며 추론합니다.
각 case별 per-channel RMSE를 수집해 통계를 생성합니다.

In [ ]:
import json
import subprocess
import textwrap
from pathlib import Path

import numpy as np

ROOT = globals().get("ROOT", Path.cwd())
CONTAINER_NAME = globals().get("CONTAINER_NAME", "motor_compare")
LOG_DIR = globals().get("LOG_DIR", ROOT / "logs" / "tutorial")
LOG_DIR.mkdir(parents=True, exist_ok=True)

FULL_CASE_INDICES = globals().get("FULL_CASE_INDICES", list(range(40)))
FULL_SOURCE_FILE_TYPES = globals().get("FULL_SOURCE_FILE_TYPES", ["OnLoadTorque"])
FULL_CKPT_PATH = globals().get(
    "FULL_CKPT_PATH",
    ROOT / "results" / "symm_mgn_doe_full40_onloadtorque.pt",
)
FULL_INFER_DIR = ROOT / "results" / "full40_infer"
FULL_INFER_DIR.mkdir(parents=True, exist_ok=True)
FULL_INFER_LOG = "12_full40_infer.log"
FULL_INFER_STEPS = [1, 2, 3]

FULL_INFER_SCRIPT = textwrap.dedent(
    """
    import json
    import os
    import sys
    import time
    from pathlib import Path

    import numpy as np
    import torch

    os.chdir("/workspace/app")
    sys.path.insert(0, "/workspace/app")

    from infer_phase1_pbc import (
        CHANNEL_ORDER,
        compute_per_channel_metrics,
        load_symm_mgn,
        run_inference,
        save_infer_npz,
    )
    from phase1_static.motor_dataset import build_samples_from_doe_manifest

    CASE_INDICES = __CASE_INDICES__
    STEP_SLOTS = __STEP_SLOTS__
    SOURCE_FILE_TYPES = __SOURCE_FILE_TYPES__
    CKPT_PATH = Path("/workspace/app/results/__CKPT_NAME__")
    OUT_DIR = Path("/workspace/app/results/full40_infer")

    script_t0 = time.time()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    model, ckpt = load_symm_mgn(CKPT_PATH, device)
    channel_order = ckpt.get("channel_order", CHANNEL_ORDER)
    channel_keys = [name.lower() for name in channel_order]
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    summaries = []
    failed = []

    for loop_idx, case_idx in enumerate(CASE_INDICES, start=1):
        samples = build_samples_from_doe_manifest(
            "/workspace/app/doe_data",
            max_steps_per_case=max(STEP_SLOTS),
            case_indices=[case_idx],
            source_file_types=SOURCE_FILE_TYPES,
        )
        ordered_samples = sorted(
            samples,
            key=lambda sample: int(sample.get("step_index", -1)),
        )

        for slot_idx, actual_sample in zip(STEP_SLOTS, ordered_samples[: len(STEP_SLOTS)]):
            actual_step_idx = int(actual_sample.get("step_index", -1))

            t0 = time.time()
            result = run_inference(model, [actual_sample], device, batch_size=1)
            elapsed = time.time() - t0
            metrics = compute_per_channel_metrics(result)

            out_path = OUT_DIR / f"full40_case{case_idx:04d}_step{slot_idx}.npz"
            meta = {
                "model_name": "SymMGN",
                "ckpt": str(CKPT_PATH),
                "case_idx": case_idx,
                "step_slot": slot_idx,
                "step_idx": actual_step_idx,
                "n_samples": 1,
                "elapsed_s": round(elapsed, 3),
                "elapsed_ms": round(elapsed * 1000.0, 1),
                "channel_order": channel_order,
                "pbc_enabled": True,
                "pbc_rotation_deg": -45.0,
                "source_file_types": SOURCE_FILE_TYPES,
                "source_file_name": actual_sample.get("source_file_name", ""),
                "step_semantics": actual_sample.get("step_semantics", ""),
                "time_s": float(actual_sample.get("time_s", 0.0)),
                "rotate_step": float(actual_sample.get("rotate_step", 0.0)),
            }
            save_infer_npz(out_path, result, metrics, meta)

            summary = {
                "case_idx": case_idx,
                "step_slot": slot_idx,
                "step_idx": actual_step_idx,
                "elapsed_ms": round(elapsed * 1000.0, 1),
                "size_kb": round(out_path.stat().st_size / 1024.0, 1),
            }
            for channel_name in channel_keys:
                summary[f"rmse_{channel_name}"] = round(
                    float(metrics[f"rmse_{channel_name}"]),
                    6,
                )
            summaries.append(summary)

        if len(ordered_samples) < len(STEP_SLOTS):
            for slot_idx in STEP_SLOTS[len(ordered_samples):]:
                failed.append(
                    {
                        "case_idx": case_idx,
                        "step_slot": slot_idx,
                        "reason": "missing_sample",
                    }
                )

        if loop_idx % 10 == 0 or loop_idx == len(CASE_INDICES):
            print(json.dumps({"progress": f"{loop_idx}/{len(CASE_INDICES)}"}, ensure_ascii=False))

    aggregate_rmse = {}
    for channel_name in channel_keys:
        values = [
            float(item[f"rmse_{channel_name}"])
            for item in summaries
            if f"rmse_{channel_name}" in item
        ]
        if values:
            values_arr = np.asarray(values, dtype=float)
            aggregate_rmse[channel_name] = {
                "mean": round(float(values_arr.mean()), 6),
                "std": round(float(values_arr.std()), 6),
                "max": round(float(values_arr.max()), 6),
            }

    peak_gpu_mb = None
    if torch.cuda.is_available():
        peak_gpu_mb = round(torch.cuda.max_memory_allocated(device) / (1024.0 ** 2), 1)

    avg_elapsed_ms = None
    if summaries:
        avg_elapsed_ms = round(
            float(np.mean([item["elapsed_ms"] for item in summaries])),
            1,
        )

    print(
        json.dumps(
            {
                "requested_cases": len(CASE_INDICES),
                "requested_step_slots": STEP_SLOTS,
                "saved_count": len(summaries),
                "failed": failed,
                "wall_time_s": round(time.time() - script_t0, 3),
                "avg_elapsed_ms": avg_elapsed_ms,
                "peak_gpu_mb": peak_gpu_mb,
                "aggregate_rmse": aggregate_rmse,
            },
            ensure_ascii=False,
        )
    )
    """
).replace("__CASE_INDICES__", json.dumps(FULL_CASE_INDICES)).replace(
    "__STEP_SLOTS__", json.dumps(FULL_INFER_STEPS)
).replace(
    "__SOURCE_FILE_TYPES__", json.dumps(FULL_SOURCE_FILE_TYPES)
).replace(
    "__CKPT_NAME__", FULL_CKPT_PATH.name
)

cp_full_infer = subprocess.run(
    ["docker", "exec", CONTAINER_NAME, "python", "-c", FULL_INFER_SCRIPT],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)

save_log_fn = globals().get("save_log")
if callable(save_log_fn):
    save_log_fn(FULL_INFER_LOG, cp_full_infer)
else:
    (LOG_DIR / FULL_INFER_LOG).write_text(
        (cp_full_infer.stdout or "") + "\n" + (cp_full_infer.stderr or ""),
        encoding="utf-8",
    )

stdout_lines = [
    line.strip()
    for line in (cp_full_infer.stdout or "").splitlines()
    if line.strip()
]
parsed_lines = []
for line in stdout_lines:
    try:
        parsed_lines.append(json.loads(line))
    except json.JSONDecodeError:
        print(line)

summary = next((item for item in reversed(parsed_lines) if "saved_count" in item), {})
progress_updates = [item["progress"] for item in parsed_lines if "progress" in item]

print("=== Full 40-case Inference Summary ===")
print(
    json.dumps(
        {
            "returncode": cp_full_infer.returncode,
            "progress_updates": progress_updates,
            **summary,
            "log_file": str(LOG_DIR / FULL_INFER_LOG),
        },
        indent=2,
        ensure_ascii=False,
    )
)

expected_count = len(FULL_CASE_INDICES) * len(FULL_INFER_STEPS)
if cp_full_infer.returncode != 0:
    print("=== infer stderr tail ===")
    print("\n".join((cp_full_infer.stderr or "").splitlines()[-40:]))
    raise RuntimeError("Full 40-case SymMGN 추론 실패. 12_full40_infer.log를 확인하세요.")

if summary.get("saved_count", 0) != expected_count:
    raise RuntimeError(
        f"Step별 NPZ 저장 수가 예상과 다릅니다: "
        f"{summary.get('saved_count', 0)} / {expected_count}"
    )

if summary.get("failed"):
    raise RuntimeError(
        "일부 case/step 저장이 실패했습니다. "
        "12_full40_infer.log를 확인하세요."
    )

## 13) Full 40-case GT vs Pred 시각화

대표 case 4개를 골라 GT vs Pred scatter를 비교합니다.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = globals().get("ROOT", Path.cwd())
FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
VIS_CASE_INDICES = globals().get("VIS_CASE_INDICES", [0, 1, 2, 3])
VIS_STEP_INDICES = globals().get("FULL_INFER_STEPS", [1, 2, 3])

channels = ["bx", "by", "a", "je"]
labels = ["Bx", "By", "A", "Je"]

for case_idx in VIS_CASE_INDICES:
    for step_idx in VIS_STEP_INDICES:
        npz_path = FULL_INFER_DIR / f"full40_case{case_idx:04d}_step{step_idx}.npz"
        if not npz_path.exists():
            print(f"case {case_idx} step {step_idx}: NPZ 없음, skip")
            continue

        arr = np.load(npz_path, allow_pickle=True)
        meta = json.loads(arr["meta"].item()) if "meta" in arr.files else {}
        actual_step_idx = int(meta.get("step_idx", step_idx))
        pos_x = arr["pos_x"]
        pos_y = arr["pos_y"]

        fig, axes = plt.subplots(2, len(channels), figsize=(22, 7))
        fig.suptitle(
            f"case {case_idx:04d} - step {actual_step_idx} GT (top) vs Pred (bottom)"
            ,fontsize=13,
        )

        for col_idx, (ch, label) in enumerate(zip(channels, labels)):
            gt = arr[f"gt_{ch}"]
            pred = arr[f"pred_{ch}"]
            vmin = float(min(gt.min(), pred.min()))
            vmax = float(max(gt.max(), pred.max()))

            sc_gt = axes[0, col_idx].scatter(
                pos_x,
                pos_y,
                c=gt,
                cmap="RdBu_r",
                s=6,
                vmin=vmin,
                vmax=vmax,
            )
            axes[0, col_idx].set_title(f"GT {label}")
            axes[0, col_idx].set_aspect("equal")
            axes[0, col_idx].set_xticks([])
            axes[0, col_idx].set_yticks([])
            plt.colorbar(sc_gt, ax=axes[0, col_idx], fraction=0.04)

            sc_pred = axes[1, col_idx].scatter(
                pos_x,
                pos_y,
                c=pred,
                cmap="RdBu_r",
                s=6,
                vmin=vmin,
                vmax=vmax,
            )
            axes[1, col_idx].set_title(f"Pred {label}")
            axes[1, col_idx].set_aspect("equal")
            axes[1, col_idx].set_xticks([])
            axes[1, col_idx].set_yticks([])
            plt.colorbar(sc_pred, ax=axes[1, col_idx], fraction=0.04)

        plt.tight_layout()
        save_path = ROOT / "logs" / f"full40_case{case_idx:04d}_step{actual_step_idx}_gt_vs_pred.png"
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=110, bbox_inches="tight")
        plt.show()
        plt.close()
        print(
            json.dumps(
                {
                    "case_idx": case_idx,
                    "step_idx": actual_step_idx,
                    "saved_png": str(save_path),
                },
                ensure_ascii=False,
            )
        )

노트북을 켠 직후 현재 실행 상태를 확인하겠습니다. 

Notebook 요약을 가져오는 중입니다.

현재 상태를 파악했습니다. 실행 카운트 순서가 문제의 원인입니다.

**현재 실행된 셀:** 2 → 3 (DOE) → 11 → 13 → 15 → 17 → **35** 시각화  
**실행 안 된 셀:** **33** (Full 40-case 추론)

셀 35 는 이전 세션에서 만든 NPZ 파일이 디스크에 남아서 that으로 실행됐고, 셀 33 은 아직 이번 세션에서 실행되지 않았습니다. 체크포인트 파일 존재 여부와 NPZ 파일 상태를 먼저 확인하겠습니다. 

Ran terminal command: ls results\symm_mgn_doe_full40_onloadtorque.pt *>&1 ; echo "---" ; (Get-ChildItem results\full40_infer\*.npz -ErrorAction SilentlyContinue | Measure-Object).Count ; echo "NPZ count above"

원인이 명확합니다.

**현재 상태 요약:**
- 체크포인트 파일 ✅ 존재 (`symm_mgn_doe_full40_onloadtorque.pt`)
- NPZ 파일 ✅ 120개 이미 존재 (이전 세션에서 생성)
- 셀 33 ❌ 이번 세션에서 아직 실행 안 됨

---

**"요청이 실패한다"는 이유:**  
이번 세션에서 **셀 33 을 건너뛰고** 셀 35 만 실행했기 때문에, 커널 변수 `FULL_CASE_INDICES`, `FULL_CKPT_PATH`, `FULL_INFER_STEPS` 등이 정의되지 않은 채로 다른 셀을 실행하면 `NameError`가 발생합니다.

그리고 셀 2(0-B), 3(0-C) 가 실행됐다는 건 **Motor-CAD DOE 셀을 먼저 돌렸다**는 것인데, 이는 일반적으로 Phase 1 추론만 할 때는 필요 없는 순서입니다.

---

**지금 해야 할 정확한 실행 순서 (Phase 1 추론/시각화 재실행):**

| 순서 | 셀 번호 | 설명 |
|------|---------|------|
| 1 | **셀 11** | ROOT, CONTAINER_NAME, LOG_DIR 설정 |
| 2 | **셀 13** | `run_local`, `run_docker`, `save_log` 함수 정의 |
| 3 | **셀 33** | Full 40-case 추론 (이미 120개 NPZ 있으므로 덮어쓰기) |
| 4 | **셀 35** | GT vs Pred 시각화 |

셀 11, 13 은 이미 실행됐으므로, **그냥 셀 33 을 실행**하면 됩니다. 이미 커널에 `ROOT`, `CONTAINER_NAME`, `LOG_DIR`, `save_log` 가 있습니다.

지금 셀 33 을 실행해 드릴까요? 



## 13-B) Full 40-case |GT - Pred| Error Map

동일 대표 case 4개에 대해 채널별 absolute error를 시각화합니다.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
VIS_CASE_INDICES = [0, 10, 20, 30]  # 대표 4개 case
channels = ["bx", "by", "a", "j", "je"]
labels = ["Bx", "By", "A", "J", "Je"]

fig, axes = plt.subplots(
    len(VIS_CASE_INDICES),
    len(channels),
    figsize=(20, 4 * len(VIS_CASE_INDICES)),
)
axes = np.atleast_2d(axes)
fig.suptitle("Full 40-case SymMGN — |GT - Pred| error map", fontsize=14)

for row_idx, case_idx in enumerate(VIS_CASE_INDICES):
    npz_path = FULL_INFER_DIR / f"full40_case{case_idx:04d}_step1.npz"
    if not npz_path.exists():
        for col_idx in range(len(channels)):
            axes[row_idx, col_idx].text(
                0.5,
                0.5,
                f"case {case_idx}\nNPZ 없음",
                ha="center",
                va="center",
            )
            axes[row_idx, col_idx].set_axis_off()
        continue

    arr = np.load(npz_path, allow_pickle=True)
    pos_x, pos_y = arr["pos_x"], arr["pos_y"]

    for col_idx, (ch, label) in enumerate(zip(channels, labels)):
        gt = arr[f"gt_{ch}"]
        pred = arr[f"pred_{ch}"]
        error = np.abs(gt - pred)
        vmax_err = float(np.percentile(error, 95))

        sc = axes[row_idx, col_idx].scatter(
            pos_x,
            pos_y,
            c=error,
            cmap="hot_r",
            s=6,
            vmin=0,
            vmax=vmax_err,
        )
        axes[row_idx, col_idx].set_aspect("equal")
        axes[row_idx, col_idx].set_xticks([])
        axes[row_idx, col_idx].set_yticks([])
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(f"|GT-Pred| {label}")
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(f"case {case_idx:04d}")
        plt.colorbar(sc, ax=axes[row_idx, col_idx], fraction=0.04)

plt.tight_layout()
vis_path = ROOT / "logs" / "full40_gt_vs_pred_error.png"
vis_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(vis_path, dpi=110, bbox_inches="tight")
plt.show()
plt.close()
print(f"saved: {vis_path}")

## 13-C) Anti-Periodic 대칭을 이용한 Full Motor 복원

1/8 섹터 추론 결과를 8회 회전 + anti-periodic sign flip으로 full 360° 모터 단면을 복원합니다.

**변환 규칙 (섹터 k = 0…7, α = 45°):**
- 위치: $(x, y)$ → 회전 $k\alpha$
- 스칼라 (A, J): $(-1)^k$ 부호 반전
- 벡터 (Bx, By): $(-1)^k$ × 회전 행렬 적용

$$\begin{bmatrix} B_x' \\ B_y' \end{bmatrix} = (-1)^k \begin{bmatrix} \cos k\alpha & -\sin k\alpha \\ \sin k\alpha & \cos k\alpha \end{bmatrix} \begin{bmatrix} B_x \\ B_y \end{bmatrix}$$

## 14) Phase 1 완료 Evidence Summary

Phase 1 (Static SymMGN with PBC) 완료 체크리스트:

| 항목 | 상태 |
|------|------|
| Contract 경계 테스트 통과 | ✅ 위 셀에서 확인 |
| PBC 경계/계약 호스트 테스트 | ✅ 위 셀에서 확인 |
| Overfit-Single 게이트 통과 | ✅ 위 셀에서 확인 |
| PBC 가시화 (학습 전) | ✅ 위 셀에서 확인 |
| 3-case Smoke Test 통과 | ✅ 위 셀에서 확인 |
| Full 40-case 학습 완료 | ⬜ 위 셀 실행 후 체크 |
| Full 40-case 추론 + RMSE 통계 | ⬜ 위 셀 실행 후 체크 |
| GT vs Pred 시각화 | ⬜ 위 셀 실행 후 체크 |

Phase 1이 완료되면 `phase2_tutorial.ipynb`로 진행합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FULL_INFER_DIR = globals().get("FULL_INFER_DIR", ROOT / "results" / "full40_infer")
FULL_VIS_CASE = 0

SECTOR_COUNT = 8
SECTOR_ANGLE_DEG = 45.0


def reconstruct_full_motor(pos_x, pos_y, fields_dict, n_sectors=8, angle_deg=45.0):
    all_x, all_y = [], []
    all_fields = {key: [] for key in fields_dict}

    for sector_idx in range(n_sectors):
        rad = np.radians(sector_idx * angle_deg)
        cos_k = np.cos(rad)
        sin_k = np.sin(rad)
        sign = (-1.0) ** sector_idx

        rot_x = pos_x * cos_k - pos_y * sin_k
        rot_y = pos_x * sin_k + pos_y * cos_k
        all_x.append(rot_x)
        all_y.append(rot_y)

        bx_orig = fields_dict["bx"]
        by_orig = fields_dict["by"]
        all_fields["bx"].append(sign * (bx_orig * cos_k - by_orig * sin_k))
        all_fields["by"].append(sign * (bx_orig * sin_k + by_orig * cos_k))
        all_fields["a"].append(sign * fields_dict["a"])
        all_fields["je"].append(sign * fields_dict["je"])

    full_x = np.concatenate(all_x)
    full_y = np.concatenate(all_y)
    full_fields = {key: np.concatenate(values) for key, values in all_fields.items()}
    return full_x, full_y, full_fields


npz_path = FULL_INFER_DIR / f"full40_case{FULL_VIS_CASE:04d}_step1.npz"
if not npz_path.exists():
    raise FileNotFoundError(f"NPZ not found: {npz_path}")

arr = np.load(npz_path, allow_pickle=True)
sector_x = arr["pos_x"]
sector_y = arr["pos_y"]

channels = ["bx", "by", "a", "je"]
labels = ["Bx", "By", "A", "Je"]
gt_fields = {channel: arr[f"gt_{channel}"] for channel in channels}
pred_fields = {channel: arr[f"pred_{channel}"] for channel in channels}

gt_full_x, gt_full_y, gt_full = reconstruct_full_motor(sector_x, sector_y, gt_fields)
pred_full_x, pred_full_y, pred_full = reconstruct_full_motor(sector_x, sector_y, pred_fields)

fig, axes = plt.subplots(2, len(channels), figsize=(24, 10))
fig.suptitle(
    f"Full Motor (8 sectors) — case {FULL_VIS_CASE:04d} GT (top) vs Pred (bottom)",
    fontsize=14,
)

for col_idx, (channel, label) in enumerate(zip(channels, labels)):
    gt_vals = gt_full[channel]
    pred_vals = pred_full[channel]
    vmin = float(min(gt_vals.min(), pred_vals.min()))
    vmax = float(max(gt_vals.max(), pred_vals.max()))

    sc_gt = axes[0, col_idx].scatter(
        gt_full_x,
        gt_full_y,
        c=gt_vals,
        cmap="RdBu_r",
        s=1.5,
        vmin=vmin,
        vmax=vmax,
    )
    axes[0, col_idx].set_title(f"GT {label}")
    axes[0, col_idx].set_aspect("equal")
    axes[0, col_idx].set_xticks([])
    axes[0, col_idx].set_yticks([])
    plt.colorbar(sc_gt, ax=axes[0, col_idx], fraction=0.046)

    sc_pred = axes[1, col_idx].scatter(
        pred_full_x,
        pred_full_y,
        c=pred_vals,
        cmap="RdBu_r",
        s=1.5,
        vmin=vmin,
        vmax=vmax,
    )
    axes[1, col_idx].set_title(f"Pred {label}")
    axes[1, col_idx].set_aspect("equal")
    axes[1, col_idx].set_xticks([])
    axes[1, col_idx].set_yticks([])
    plt.colorbar(sc_pred, ax=axes[1, col_idx], fraction=0.046)

plt.tight_layout()
save_path = ROOT / "logs" / f"full_motor_case{FULL_VIS_CASE:04d}_gt_vs_pred.png"
save_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(save_path, dpi=130, bbox_inches="tight")
plt.show()
plt.close()
print(f"saved: {save_path}")
print(f"total nodes: {len(gt_full_x)} ({len(sector_x)} x {SECTOR_COUNT} sectors)")